In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x8

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- الخلية 22: المحرك البلاتيني النهائي (V36: Dynamic Length + Safe Mode + Original Calibration) ---
# هدفها:
# 1) تمنع Notebook Threw Exception (صور تالفة/ضخمة/OOM)
# 2) تمنع كارثة 5000 ثابت (Dynamic Length من القالب parquet)
# 3) تحافظ على بايبلاينك الأصلي (grid calibration + filters) بدون auto_scale أو تخمينات

import gc, os, csv, glob
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# ---------------------------
# 0) إعدادات ثابتة
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"
SUBMISSION_FILE = "submission.csv"

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_TO_IDX = {name: i for i, name in enumerate(LEAD_NAMES)}

# حماية عامة
MAX_UNET_WIDTH = 2048       # لمنع انفجار الذاكرة
MAX_BATCH_PAD_W = 3000      # سقف padding النهائي
MIN_CROP_PIXELS = 100       # تجاهل crop الفارغ
CACHE_PATIENTS = 12         # كاش مرضى (LRU)

print(f"⚡ Device: {DEVICE}")

# ---------------------------
# 1) قراءة القالب (Parquet) + حساب الطول لكل مريض (Dynamic Length)
# ---------------------------
if not os.path.exists(SAMPLE_PARQUET_PATH):
    raise FileNotFoundError(f"❌ Missing template parquet: {SAMPLE_PARQUET_PATH}")

print("📦 Reading Parquet template ids...")
template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = template["id"].astype(str).to_numpy()
del template
gc.collect()

print(f"✅ Template rows: {len(template_ids):,}")
print("🧠 Scanning template for dynamic lengths per patient...")

patient_lengths = {}
for rid in tqdm(template_ids, desc="Scan Lengths"):
    try:
        parts = rid.rsplit("_", 2)
        if len(parts) != 3:
            continue
        pid_part, idx_part, _ = parts
        pid = str(pid_part).strip()
        if pid.endswith(".0"):
            pid = pid[:-2]
        idx = int(idx_part)
        cur = patient_lengths.get(pid, 0)
        if idx + 1 > cur:
            patient_lengths[pid] = idx + 1
    except:
        continue

print(f"✅ Dynamic lengths computed for {len(patient_lengths):,} patients.")

# ---------------------------
# 2) fs_map من test.csv
# ---------------------------
fs_map = {}
if os.path.exists(TEST_CSV_PATH):
    try:
        tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
        cols_lower = {c.lower(): c for c in tdf.columns}
        id_col = next((cols_lower[c] for c in cols_lower if "id" in c), None)
        fs_col = next((cols_lower[c] for c in cols_lower if "fs" in c), None)
        if id_col and fs_col:
            fs_map = dict(zip(tdf[id_col].astype(str), tdf[fs_col].astype(str)))
            print(f"✅ fs_map loaded: {len(fs_map):,} items")
        else:
            print("⚠️ fs columns not found; default fs=500 will be used.")
    except Exception as e:
        print(f"⚠️ Failed reading test.csv: {e}")

# ---------------------------
# 3) فهرسة الصور
# ---------------------------
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path_safe(pid):
    pid_s = clean_pid(pid)
    if pid_s in path_map:
        return path_map[pid_s]
    try:
        pid_int = str(int(float(pid_s)))
        return path_map.get(pid_int)
    except:
        return None

# ---------------------------
# 4) تحميل الموديلات (Offline)
# ---------------------------
print("⚙️ Loading models (offline)...")
YOLO_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

yolo_model = None
if os.path.exists(YOLO_PATH):
    try:
        yolo_model = YOLO(YOLO_PATH)
        print(f"✅ YOLO loaded: {YOLO_PATH}")
    except Exception as e:
        print(f"⚠️ YOLO load failed: {e}")

unet_model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    decoder_attention_type="scse"
)
if os.path.exists(UNET_PATH):
    try:
        unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
        print(f"✅ UNet loaded: {UNET_PATH}")
    except Exception as e:
        print(f"⚠️ UNet load failed: {e}")
else:
    print("⚠️ UNet checkpoint not found; outputs may be poor.")

unet_model.to(DEVICE).eval()

# ---------------------------
# 5) دوال مساعدة (Safe + Original Logic)
# ---------------------------
def apply_high_pass_filter(data, cutoff=0.5, fs=500.0, order=5):
    try:
        if len(data) < order * 3:
            return data
        nyq = 0.5 * float(fs)
        if nyq <= 0:
            return data
        normal_cutoff = cutoff / nyq
        if not (0 < normal_cutoff < 1):
            return data
        b, a = butter(order, normal_cutoff, btype="high", analog=False)
        return filtfilt(b, a, data)
    except:
        return data

def smart_einthoven_fix(leads):
    try:
        if 'I' in leads and 'II' in leads and 'III' in leads:
            l = min(len(leads['I']), len(leads['II']), len(leads['III']))
            I = leads['I'][:l]
            II = leads['II'][:l]
            III = leads['III'][:l]
            residual = (I + III) - II
            leads['II'][:l] += residual / 3.0
            leads['I'][:l]  -= residual / 3.0
            leads['III'][:l]-= residual / 3.0
    except:
        pass
    return leads

def robust_multi_point_calibration(crop, default_val=25.0):
    # نفس الفكرة عندك + حماية إضافية
    try:
        if crop is None or crop.size == 0:
            return float(default_val)
        gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
        if gray.size == 0 or np.std(gray) < 3:
            return float(default_val)

        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)

        edges_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        edges_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sx = np.sum(np.abs(edges_x), axis=0)
        sy = np.sum(np.abs(edges_y), axis=1)

        peaks_x, _ = find_peaks(sx, distance=10, prominence=50)
        peaks_y, _ = find_peaks(sy, distance=10, prominence=50)

        grid_sizes = []
        if len(peaks_x) > 3:
            dx = np.diff(peaks_x)
            grid_sizes.extend(dx[(dx > 10) & (dx < 100)])
        if len(peaks_y) > 3:
            dy = np.diff(peaks_y)
            grid_sizes.extend(dy[(dy > 10) & (dy < 100)])

        if len(grid_sizes) >= 5:
            grid_sizes = np.array(grid_sizes, dtype=np.float32)
            q1 = np.percentile(grid_sizes, 25)
            q3 = np.percentile(grid_sizes, 75)
            iqr = q3 - q1
            clean = grid_sizes[(grid_sizes >= q1 - 1.5*iqr) & (grid_sizes <= q3 + 1.5*iqr)]
            if len(clean) > 0:
                return float(0.6*np.median(clean) + 0.4*np.mean(clean))
    except:
        pass
    return float(default_val)

def advanced_perspective_correction(image):
    # نسخة آمنة من دالتك
    try:
        if image is None:
            return None
        h, w = image.shape[:2]
        if h < 32 or w < 32:
            return image
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        small = cv2.resize(gray, (max(32, w//2), max(32, h//2)))
        edges = cv2.Canny(small, 50, 150)
        lines = cv2.HoughLinesP(
            edges, 1, np.pi/180,
            threshold=80,
            minLineLength=max(10, small.shape[1]//8),
            maxLineGap=10
        )
        if lines is None:
            return image

        angles = []
        for l in lines:
            x1,y1,x2,y2 = l[0]
            ang = np.degrees(np.arctan2(y2-y1, x2-x1))
            if abs(ang) < 15:
                angles.append(ang)
        if len(angles) < 5:
            return image

        median_ang = float(np.median(angles))
        if abs(median_ang) > 0.5:
            M = cv2.getRotationMatrix2D((w//2, h//2), median_ang, 1.0)
            return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        return image
    except:
        return image

def preprocess_remove_grid_rgb(image_rgb):
    try:
        if image_rgb is None:
            return None
        hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
        mask = (
            cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
            cv2.inRange(hsv, (170, 50, 50), (180, 255, 255))
        )
        out = image_rgb.copy()
        out[mask > 0] = (255, 255, 255)
        return out
    except:
        return image_rgb

def get_yolo_crops_with_fallback(image, model):
    h, w = image.shape[:2]
    crops = [None] * 12

    if model is not None:
        try:
            results = model.predict(image, verbose=False, conf=0.15, imgsz=640, device=DEVICE)
            if results and results[0].boxes is not None:
                boxes = results[0].boxes.data.detach().cpu().numpy()
                best = {}
                for b in boxes:
                    cls_id = int(b[5])
                    conf = float(b[4])
                    if 0 <= cls_id < 12:
                        if cls_id not in best or conf > best[cls_id][0]:
                            best[cls_id] = (conf, b[:4])

                for cid, (_, xyxy) in best.items():
                    x1,y1,x2,y2 = map(int, xyxy)
                    x1,y1 = max(0,x1), max(0,y1)
                    x2,y2 = min(w,x2), min(h,y2)
                    if x2 > x1 and y2 > y1:
                        crops[cid] = image[max(0,y1-5):min(h,y2+5), max(0,x1-5):min(w,x2+5)]

                if sum(c is not None for c in crops) >= 8:
                    return crops
        except:
            pass

    # fallback grid split
    rh, cw = h//4, w//3
    if rh < 5 or cw < 5:
        return crops
    return [image[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]

def batch_predict_unet(crops, model):
    # نسخة آمنة + clamp width + anti-oom
    if not crops:
        return [], [], []
    target_h = 256
    processed = []
    scales = []
    shapes = []

    try:
        for img in crops:
            if img is None:
                continue
            h, w = img.shape[:2]
            if h <= 0 or w <= 0:
                continue

            scale = target_h / float(h)
            new_w = int(max(1, w * scale))

            # clamp width لتجنب OOM
            if new_w > MAX_UNET_WIDTH:
                new_w = MAX_UNET_WIDTH
                # ملاحظة: نبقي scale كما هي لربط pixels->mm نسبيًا
                # (بدل إعادة حسابها بشكل قد يكسر المنطق)
            img_rz = cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)
            tens = torch.from_numpy(img_rz).permute(2,0,1).float() / 255.0
            processed.append(tens)
            scales.append(scale)
            shapes.append((target_h, new_w))

        if not processed:
            return [], [], []

        max_w = max(s[1] for s in shapes)
        max_w = int(np.ceil(max_w / 32) * 32)
        if max_w > MAX_BATCH_PAD_W:
            max_w = MAX_BATCH_PAD_W

        batch = torch.zeros((len(processed), 3, target_h, max_w), dtype=torch.float32, device=DEVICE)
        for i, t in enumerate(processed):
            w_curr = min(t.shape[2], max_w)
            batch[i, :, :, :w_curr] = t[:, :, :w_curr].to(DEVICE)

        with torch.no_grad():
            p1 = torch.sigmoid(model(batch))
            p2 = torch.sigmoid(model(torch.flip(batch, [3])))
            p = (p1 + torch.flip(p2, [3])) / 2.0

        p = p.detach().cpu().numpy()
        prob_maps = [p[i, 0, :shapes[i][0], :shapes[i][1]] for i in range(len(processed))]
        return prob_maps, scales, shapes

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return [], [], []
        raise
    except:
        return [], [], []

def fast_viterbi(prob_map):
    # نسخة argmax + smoothing آمنة
    try:
        H, W = prob_map.shape
        if H <= 0 or W <= 0:
            return np.zeros((W,), dtype=np.float32)

        path = np.argmax(prob_map, axis=0).astype(np.float32)

        # savgol يحتاج odd window <= W
        if W >= 11:
            win = 11
            if win % 2 == 0:
                win += 1
            path = savgol_filter(path, win, 2)
        return (H - path).astype(np.float32)
    except:
        return np.zeros((prob_map.shape[1],), dtype=np.float32)

# ---------------------------
# 6) المعالجة الأساسية لكل مريض (مع Dynamic Length)
# ---------------------------
patient_cache = OrderedDict()

def compute_patient_leads(pid_clean_str, target_len):
    # target_len ديناميكي من القالب (2500/5000/غيره)
    default_data = np.zeros((12, target_len), dtype=np.float32)

    path = get_image_path_safe(pid_clean_str)
    if not path:
        return default_data

    # fs
    fs_val = 500.0
    try:
        fs_val = float(fs_map.get(pid_clean_str, 500.0))
    except:
        fs_val = 500.0

    try:
        img = cv2.imread(path)
        if img is None:
            return default_data
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = advanced_perspective_correction(img)
        global_grid = robust_multi_point_calibration(img, 25.0)

        crops = get_yolo_crops_with_fallback(img, yolo_model)

        clean_crops = []
        valid_idx = []
        for i, crop in enumerate(crops):
            if crop is not None and crop.size > MIN_CROP_PIXELS:
                clean_crops.append(preprocess_remove_grid_rgb(crop))
                valid_idx.append(i)

        if not clean_crops:
            return default_data

        prob_maps, scales, _shapes = batch_predict_unet(clean_crops, unet_model)
        if not prob_maps:
            return default_data

        leads_dict = {}
        for j, real_idx in enumerate(valid_idx):
            if j >= len(prob_maps):
                break

            prob = prob_maps[j]
            scale = scales[j]
            lname = LEAD_NAMES[real_idx]

            local_grid = robust_multi_point_calibration(clean_crops[j], default_val=global_grid)
            raw = fast_viterbi(prob)

            # ORIGINAL mV conversion logic (grid*scale*10)
            g_sc = float(local_grid) * float(scale)
            divider = (g_sc * 10.0) if g_sc > 1.0 else 250.0

            sig_mv = (raw - np.median(raw)) / float(divider)
            sig_mv = np.nan_to_num(sig_mv, nan=0.0, posinf=0.0, neginf=0.0)

            if len(sig_mv) > 15:
                sig_mv = savgol_filter(sig_mv, 11, 3)
            if len(sig_mv) > 20:
                sig_mv = apply_high_pass_filter(sig_mv, cutoff=0.5, fs=fs_val)

            if len(sig_mv) > 0:
                # THE KEY FIX: resample to target_len (dynamic)
                sig_rs = resample(sig_mv, target_len).astype(np.float32)
                sig_rs = np.nan_to_num(sig_rs, nan=0.0, posinf=0.0, neginf=0.0)
                leads_dict[lname] = sig_rs

        leads_dict = smart_einthoven_fix(leads_dict)

        final = np.zeros((12, target_len), dtype=np.float32)
        for i, l in enumerate(LEAD_NAMES):
            if l in leads_dict:
                final[i] = leads_dict[l]

        return final

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return default_data
        # أي RuntimeError آخر: لا تسقط النوتبوك
        torch.cuda.empty_cache()
        return default_data
    except:
        torch.cuda.empty_cache()
        return default_data

# ---------------------------
# 7) الكتابة Streaming وفق القالب (id,value) + Cache
# ---------------------------
print("🚀 Writing submission.csv (Safe Streaming + Dynamic Length)...")

with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for row_id in tqdm(template_ids, desc="Processing Rows"):
        try:
            parts = row_id.rsplit("_", 2)
            if len(parts) != 3:
                writer.writerow([row_id, "0.0000"])
                continue

            pid_part, idx_part, lead_part = parts
            pid = clean_pid(pid_part)
            idx = int(idx_part)

            target_len = int(patient_lengths.get(pid, 5000))
            if target_len <= 0:
                target_len = 5000

            # Cache
            if pid in patient_cache:
                leads_mtx = patient_cache[pid]
                patient_cache.move_to_end(pid)
            else:
                leads_mtx = compute_patient_leads(pid, target_len)
                patient_cache[pid] = leads_mtx
                if len(patient_cache) > CACHE_PATIENTS:
                    patient_cache.popitem(last=False)

            lead_idx = LEAD_TO_IDX.get(lead_part, 0)
            val = 0.0
            if 0 <= idx < target_len and 0 <= lead_idx < 12:
                v = float(leads_mtx[lead_idx][idx])
                if np.isfinite(v):
                    val = v

            writer.writerow([row_id, f"{val:.4f}"])

        except:
            writer.writerow([row_id, "0.0000"])

# Cleanup
del patient_cache
gc.collect()
torch.cuda.empty_cache()

print("✅ Done. submission.csv created with correct template rows + dynamic per-patient length.")


⚡ Device: cuda
📦 Reading Parquet template ids...
✅ Template rows: 75,000
🧠 Scanning template for dynamic lengths per patient...


Scan Lengths: 100%|██████████| 75000/75000 [00:00<00:00, 909114.76it/s]

✅ Dynamic lengths computed for 2 patients.
✅ fs_map loaded: 2 items
🗂️ Indexing images...


✅ Indexed images: 8,795
⚙️ Loading models (offline)...
✅ YOLO loaded: /kaggle/input/ecg-final-models/best.pt
✅ UNet loaded: /kaggle/input/ecg-final-models/best_model_phase10.pth
🚀 Writing submission.csv (Safe Streaming + Dynamic Length)...


Processing Rows: 100%|██████████| 75000/75000 [00:03<00:00, 21356.43it/s]


✅ Done. submission.csv created with correct template rows + dynamic per-patient length.


In [5]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.97 MB
🎉 Ready to Submit.
